In [67]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Load data

In [68]:
df = pd.read_csv('datasets/omi_estimate/omi_estimate.csv')

In [69]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
import pandas as pd
import plotly.io as pio

pio.renderers.default = 'vscode'

df_trento = df[df['mun_name'] == 'TRENTO'].sort_values('year_semester')

zones      = sorted(df_trento['zone'].unique().tolist())
types      = sorted(df_trento['type'].unique().tolist())
conditions = sorted(df_trento['condition'].unique().tolist())

# --- Widgets ---
dropdown_zone = widgets.Dropdown(
    options=[('Average (all zones)', 'average')] + [(f'{z}', z) for z in zones],
    value='average',
    description='Zone:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

dropdown_type = widgets.Dropdown(
    options=[(t, t) for t in types],
    value='Residential housing',   # default
    description='Property type:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

dropdown_condition = widgets.Dropdown(
    options=[('Average (all conditions)', 'average')] + [(c.capitalize(), c) for c in conditions],
    value='average',               # default
    description='Condition:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

output = widgets.Output()

# --- Core logic ---
def get_filtered_df(zone, prop_type, condition):
    """Apply all three filters, computing averages where needed."""
    filtered = df_trento.copy()

    # 1. Filter by property type (always a hard filter)
    filtered = filtered[filtered['type'] == prop_type]

    # 2. Filter by condition or average across all
    if condition != 'average':
        filtered = filtered[filtered['condition'] == condition]

    # 3. Filter by zone or average across all
    if zone != 'average':
        filtered = filtered[filtered['zone'] == zone]

    # Group and average to collapse any remaining duplicates
    plot_df = (
        filtered
        .groupby('year_semester')[['buy_min', 'buy_max']]
        .mean()
        .reset_index()
    )
    return plot_df


def build_subtitle(zone, prop_type, condition):
    zone_label      = 'All zones (avg)'      if zone == 'average'      else f'Zone: {zone}'
    condition_label = 'All conditions (avg)' if condition == 'average' else condition.capitalize()
    return f'{prop_type} · {zone_label} · {condition_label}'


def get_traces(plot_df):
    trace_min = go.Scatter(
        x=plot_df['year_semester'].str.replace("_"," - "),
        y=plot_df['buy_min'],
        mode='lines',
        name='buy_min',
        line=dict(color='steelblue', width=2),
        hovertemplate='Min. buying price: %{y:,.0f}<extra></extra>'
    )
    trace_max = go.Scatter(
        x=plot_df['year_semester'].str.replace("_"," - "),
        y=plot_df['buy_max'],
        mode='lines',
        name='buy_max',
        line=dict(color='tomato', width=2),
        hovertemplate='Max. buying price: %{y:,.0f}<extra></extra>'
    )
    return [trace_max, trace_min]


def build_layout(subtitle):
    return go.Layout(
        title=dict(text=f'Buy Price Trends for TRENTO<br><sup>{subtitle}</sup>'),
        xaxis=dict(
            title='Year - Semester',
            tickangle=-45,
            showspikes=True,
            spikemode='across',
            spikesnap='cursor',
            spikecolor='grey',
            spikethickness=1,
            spikedash='dash',
        ),
        yaxis=dict(title='Price (€/m²)'),
        hovermode='x unified',
        hoverdistance=50,
        spikedistance=50,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        template='plotly_white',
        height=500,
    )


def update_plot(change):
    zone      = dropdown_zone.value
    prop_type = dropdown_type.value
    condition = dropdown_condition.value

    with output:
        output.clear_output(wait=True)
        plot_df  = get_filtered_df(zone, prop_type, condition)

        if plot_df.empty:
            print(f"No data available for the current selection.")
            return

        subtitle = build_subtitle(zone, prop_type, condition)
        fig      = go.Figure(data=get_traces(plot_df), layout=build_layout(subtitle))
        fig.show()


# --- Wire all three dropdowns to the same callback ---
for dd in [dropdown_zone, dropdown_type, dropdown_condition]:
    dd.observe(update_plot, names='value')

# --- Layout ---
controls = widgets.HBox(
    [dropdown_zone, dropdown_type, dropdown_condition],
    layout=widgets.Layout(gap='16px')
)
display(widgets.VBox([controls, output]))

# Initial render
update_plot({'new': None})

In [70]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
import pandas as pd
import plotly.io as pio
from functools import partial

pio.renderers.default = 'vscode'

# --- Helpers ---
def get_available(df, reg=None, prov=None, mun=None, zone=None):
    f = df.copy()
    if reg  and reg  != 'average': f = f[f['reg_name']  == reg]
    if prov and prov != 'average': f = f[f['prov_name'] == prov]
    if mun  and mun  != 'average': f = f[f['mun_name']  == mun]
    if zone and zone != 'average': f = f[f['zone']       == zone]
    return f

def make_options(values, all_label):
    return [(all_label, 'average')] + [(str(v), v) for v in sorted(values)]

# --- Widgets ---
dropdown_region = widgets.Dropdown(
    options=make_options(df['reg_name'].dropna().unique(), 'Average (all regions)'),
    value='average',
    description='Region:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)
dropdown_province = widgets.Dropdown(
    options=make_options(df['prov_name'].dropna().unique(), 'Average (all provinces)'),
    value='average',
    description='Province:',
    disabled=True,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)
dropdown_mun = widgets.Dropdown(
    options=make_options(df['mun_name'].dropna().unique(), 'Average (all municipalities)'),
    value='average',
    description='Municipality:',
    disabled=True,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)
dropdown_zone = widgets.Dropdown(
    options=make_options(df['zone'].dropna().unique(), 'Average (all zones)'),
    value='average',
    description='Zone:',
    disabled=True,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)
dropdown_type = widgets.Dropdown(
    options=[(t, t) for t in sorted(df['type'].dropna().unique())],
    value='Residential housing',
    description='Property type:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)
dropdown_condition = widgets.Dropdown(
    options=make_options(df['condition'].dropna().unique(), 'Average (all conditions)'),
    value='average',
    description='Condition:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)

output = widgets.Output()
territorial_dropdowns = [dropdown_region, dropdown_province, dropdown_mun, dropdown_zone]
levels = ['region', 'province', 'mun', 'zone']

# --- Cascade logic ---
def update_cascade(changed_level):
    # Unobserve using stored handler objects to avoid ValueError
    for level, dd in zip(levels, territorial_dropdowns):
        dd.unobserve(handlers[level], names='value')

    # ── Step 1: reset downstream values ──────────────────────────────────
    reset_map = {
        'region':   [dropdown_province, dropdown_mun, dropdown_zone],
        'province': [dropdown_mun, dropdown_zone],
        'mun':      [dropdown_zone],
        'zone':     [],
    }
    for dd in reset_map[changed_level]:
        dd.value = 'average'

    # Read current values after reset
    reg  = dropdown_region.value
    prov = dropdown_province.value
    mun  = dropdown_mun.value
    zone = dropdown_zone.value

    # ── Step 2: refresh child options ────────────────────────────────────
    f_reg = get_available(df, reg=reg)
    dropdown_province.options = make_options(
        f_reg['prov_name'].dropna().unique(), 'Average (all provinces)')

    f_prov = get_available(df, reg=reg, prov=prov)
    dropdown_mun.options = make_options(
        f_prov['mun_name'].dropna().unique(), 'Average (all municipalities)')

    f_mun = get_available(df, reg=reg, prov=prov, mun=mun)
    dropdown_zone.options = make_options(
        f_mun['zone'].dropna().unique(), 'Average (all zones)')

    # ── Step 3: enable/disable based on parent selection ─────────────────
    dropdown_province.disabled = (reg  == 'average')
    dropdown_mun.disabled      = (prov == 'average')
    dropdown_zone.disabled     = (mun  == 'average')

    # ── Step 4: refresh type + condition from full territorial context ────
    f_full = get_available(df, reg=reg, prov=prov, mun=mun, zone=zone)

    new_types = sorted(f_full['type'].dropna().unique())
    preferred = 'Residential housing' if 'Residential housing' in new_types else (new_types[0] if new_types else None)

    # Set value BEFORE options to prevent ipywidgets auto-resetting to first item
    
    dropdown_type.value   = preferred                    # ← always force it after
    #dropdown_type.options = [(t, t) for t in new_types]

    new_conds = make_options(f_full['condition'].dropna().unique(), 'Average (all conditions)')
    dropdown_condition.options = new_conds
    if dropdown_condition.value not in [v for _, v in new_conds]:
        dropdown_condition.value = 'average'

    # Re-observe using the same stored handler objects
    for level, dd in zip(levels, territorial_dropdowns):
        dd.observe(handlers[level], names='value')

    update_plot()


def on_territorial_change(level, change):
    update_cascade(level)

# Store partials in a dict so the exact same object is used for observe/unobserve
handlers = {}
for level, dd in zip(levels, territorial_dropdowns):
    handler = partial(on_territorial_change, level)
    handlers[level] = handler
    dd.observe(handler, names='value')

for dd in [dropdown_type, dropdown_condition]:
    dd.observe(lambda change: update_plot(), names='value')


# --- Plot logic ---
def get_filtered_df():
    filtered = get_available(
        df,
        reg=dropdown_region.value,
        prov=dropdown_province.value,
        mun=dropdown_mun.value,
        zone=dropdown_zone.value
    )
    filtered = filtered[filtered['type'] == dropdown_type.value]
    if dropdown_condition.value != 'average':
        filtered = filtered[filtered['condition'] == dropdown_condition.value]

    return (
        filtered
        .groupby('year_semester')[['buy_min', 'buy_max']]
        .mean()
        .reset_index()
        .sort_values('year_semester')
    )


def build_subtitle():
    reg, prov, mun, zone = (
        dropdown_region.value, dropdown_province.value,
        dropdown_mun.value,    dropdown_zone.value
    )
    territory = (
        mun  if mun  != 'average' else
        prov if prov != 'average' else
        reg  if reg  != 'average' else
        'All regions (avg)'
    )
    zone_label      = f'Zone: {zone}'                    if zone      != 'average' else 'All zones (avg)'
    condition_label = dropdown_condition.value.capitalize() \
                      if dropdown_condition.value        != 'average' else 'All conditions (avg)'
    return f'{dropdown_type.value} · {territory} · {zone_label} · {condition_label}'


def get_traces(plot_df):
    x = plot_df['year_semester'].str.replace('_', ' - ')
    trace_max = go.Scatter(
        x=x, y=plot_df['buy_max'], mode='lines', name='buy_max',
        line=dict(color='tomato', width=2),
        hovertemplate='Max. buying price: %{y:,.0f}<extra></extra>'
    )
    trace_min = go.Scatter(
        x=x, y=plot_df['buy_min'], mode='lines', name='buy_min',
        line=dict(color='steelblue', width=2),
        hovertemplate='Min. buying price: %{y:,.0f}<extra></extra>'
    )
    return [trace_max, trace_min]


def build_layout(subtitle):
    return go.Layout(
        title=dict(text=f'Buy Price Trends<br><sup>{subtitle}</sup>'),
        xaxis=dict(
            title='Year - Semester', tickangle=-45,
            showspikes=True, spikemode='across', spikesnap='cursor',
            spikecolor='grey', spikethickness=1, spikedash='dash',
        ),
        yaxis=dict(title='Price (€/m²)'),
        hovermode='x unified',
        hoverdistance=50, spikedistance=50,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        template='plotly_white',
        height=500,
    )


def update_plot():
    with output:
        output.clear_output(wait=True)
        plot_df = get_filtered_df()
        if plot_df.empty:
            print("⚠️ No data available for the current selection.")
            return
        fig = go.Figure(data=get_traces(plot_df), layout=build_layout(build_subtitle()))
        fig.show()


# --- UI ---
row1 = widgets.HBox(
    [dropdown_region, dropdown_province, dropdown_mun, dropdown_zone],
    layout=widgets.Layout(gap='12px')
)
row2 = widgets.HBox(
    [dropdown_type, dropdown_condition],
    layout=widgets.Layout(gap='12px')
)

display(widgets.VBox([row1, row2, output]))
update_cascade('region')    # initial render — handlers dict already exists